In [16]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import mobilenet_v3_large, MobileNet_V3_Large_Weights
from PIL import Image
from torchvision.transforms import v2
from torchinfo import summary
import pandas as pd
import numpy as np
import os

**Enable cuda if available**

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [18]:
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [19]:
writer = SummaryWriter()

In [20]:
class ISIC2019(Dataset): # TODO: consider maybe removing downsampled or removing duplicates. unsure if these are necessary
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_labels.drop("UNK", axis=1, inplace=True) # remove unknown category
        self.ohe_labels = self.img_labels.iloc[:, 1:]

        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f"{self.img_labels.iloc[idx, 0]}.jpg")
        image = Image.open(img_path)
        label = np.where(self.ohe_labels==1)[1][idx]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

In [21]:
transform = v2.Compose([
    v2.Resize((224, 224)),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])

In [22]:
isic_dataset = ISIC2019(img_dir='../data/ISIC_2019_Training_Input', annotations_file='../data/ISIC_2019_Training_GroundTruth.csv', transform=transform)
train_images_size = len(isic_dataset)

In [23]:
train_size = int(train_images_size * 0.80)
test_size = train_images_size - train_size
# 80% train 20% test

isic_train, isic_test = random_split(isic_dataset, [train_size, test_size])
train_size, test_size

(20264, 5067)

In [24]:
class MobileNetV3_Baseline(nn.Module):
    def __init__(self):
        super().__init__()
        self.num_classes = 8
        self.mobilenet = mobilenet_v3_large(weights=MobileNet_V3_Large_Weights.DEFAULT) # use imagenet pretrained weights
        in_features = self.mobilenet.classifier[3].in_features
        self.mobilenet.classifier[3] = nn.Linear(in_features, self.num_classes) # change number of output categories

        nn.init.normal_(self.mobilenet.classifier[3].weight, 0, 0.01) 
        nn.init.zeros_(self.mobilenet.classifier[3].bias) # use same weight + bias init as pytorch for consistency

    def forward(self, x):
        x = self.mobilenet(x)
        return x

In [25]:
model = MobileNetV3_Baseline()

**Move model to specified device**

In [26]:
model = model.to(device=device)

In [ ]:
epochs = 100
learning_rate = 1e-4
batch_size = 32

start_epoch = 0 # epoch number of checkpoint to load

if start_epoch > 0:
    model.load_state_dict(torch.load(f"checkpoints/baseline/epoch-{start_epoch}.pth")) # load specific checkpoint

output_dir = "checkpoints/baseline"
os.makedirs(output_dir, exist_ok=True) # create baseline dir for checkpoints

# TODO - find better parameters for baseline

In [28]:
summary(model, input_size=(batch_size, 3, 224, 224)) # show model summary and all details

Layer (type:depth-idx)                                  Output Shape              Param #
MobileNetV3_Baseline                                    [32, 8]                   --
├─MobileNetV3: 1-1                                      [32, 8]                   --
│    └─Sequential: 2-1                                  [32, 960, 7, 7]           --
│    │    └─Conv2dNormActivation: 3-1                   [32, 16, 112, 112]        464
│    │    └─InvertedResidual: 3-2                       [32, 16, 112, 112]        464
│    │    └─InvertedResidual: 3-3                       [32, 24, 56, 56]          3,440
│    │    └─InvertedResidual: 3-4                       [32, 24, 56, 56]          4,440
│    │    └─InvertedResidual: 3-5                       [32, 40, 28, 28]          10,328
│    │    └─InvertedResidual: 3-6                       [32, 40, 28, 28]          20,992
│    │    └─InvertedResidual: 3-7                       [32, 40, 28, 28]          20,992
│    │    └─InvertedResidual: 3-8       

In [29]:
dataloader_train = DataLoader(isic_train, batch_size=batch_size, shuffle=True, pin_memory=True)
dataloader_test = DataLoader(isic_test, batch_size=batch_size, shuffle=False, pin_memory=True)

num_train_batches = len(dataloader_train)
num_test_batches = len(dataloader_test)

loss = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
num_train_batches, num_test_batches

(634, 159)

In [30]:
for epoch in range(start_epoch, epochs):
    train_loss = 0
    train_acc = 0

    model.train()
    for batch_idx, (train_features, train_labels) in enumerate(dataloader_train):
        train_features = train_features.to(device)
        train_labels = train_labels.to(device) # move to device

        optimizer.zero_grad()

        predictions = model(train_features)        
        predictions_labels = torch.argmax(predictions, dim=1)
        
        train_batch_acc = (predictions_labels == train_labels).sum().item() / train_features.shape[0]

        train_batch_loss = loss(predictions, train_labels)
        train_batch_loss.backward()

        optimizer.step()

        train_loss += train_batch_loss.item()
        train_acc += train_batch_acc

    print(f"Saving epoch {epoch+1}...")
    torch.save(model.state_dict(), f"checkpoints/baseline/epoch-{epoch+1}.pth") # checkpoint per epoch for safety

    test_loss = 0
    test_acc = 0

    model.eval()
    with torch.no_grad():
        for batch_idx, (test_features, test_labels) in enumerate(dataloader_test):
            test_features = test_features.to(device)
            test_labels = test_labels.to(device) # move to device
            
            predictions = model(test_features)
            predictions_labels = torch.argmax(predictions, dim=1)

            test_batch_acc = (predictions_labels == test_labels).sum().item() / test_features.shape[0]
            test_batch_loss = loss(predictions, test_labels)

            test_loss += test_batch_loss.item()
            test_acc += test_batch_acc

    train_loss /= num_train_batches
    train_acc /= num_train_batches

    test_loss /= num_test_batches
    test_acc /= num_test_batches

    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_acc, epoch)

    writer.add_scalar("Loss/test", test_loss, epoch)
    writer.add_scalar('Accuracy/test', test_acc, epoch)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81


KeyboardInterrupt: 

In [ ]:
torch.save(model.state_dict(), f"checkpoints/baseline/final_checkpoint.pth") # last checkpoint after model is done training

In [ ]:
writer.flush()